# Gradient Descent

This notebook accompanies the **ML Viz** lesson on Gradient Descent.

We start by reproducing the lesson's hand-worked example — fitting a line with mean
squared error (MSE) — then build SGD, Momentum, RMSprop, and Adam from scratch and
visualize their trajectories on a 2D loss surface.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/neural-networks/02-gradient-descent

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Worked example: fitting a line with MSE

This reproduces the lesson's by-hand derivation. The model is a line
$\hat{y} = w x + b$ and the loss is the mean squared error

$$L(w, b) = \frac{1}{n}\sum_{i=1}^{n}\bigl(w x_i + b - y_i\bigr)^2.$$

The gradients (derived via the chain rule on each residual $r_i = w x_i + b - y_i$) are

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (w x_i + b - y_i)\,x_i,
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (w x_i + b - y_i).$$

We fit three points that lie exactly on $y = 2x + 1$, starting from $w = b = 0$
with learning rate $\eta = 0.1$ — so the answer should converge to $(w, b) = (2, 1)$.

In [ ]:
# Training data: three points on the line y = 2x + 1
X = np.array([1.0, 2.0, 3.0])
Y = np.array([3.0, 5.0, 7.0])


def mse_loss(w, b):
    """Mean squared error of the line y = w*x + b on (X, Y)."""
    residuals = w * X + b - Y
    return np.mean(residuals ** 2)


def mse_grads(w, b):
    """Analytical gradients dL/dw and dL/db."""
    residuals = w * X + b - Y          # r_i = w*x_i + b - y_i
    dw = 2.0 * np.mean(residuals * X)  # (2/n) * sum(r_i * x_i)
    db = 2.0 * np.mean(residuals)      # (2/n) * sum(r_i)
    return dw, db


def fit_line(w0=0.0, b0=0.0, lr=0.1, n_steps=50):
    """Run gradient descent, recording the (w, b) path and the loss curve."""
    w, b = w0, b0
    ws, bs, losses = [w], [b], [mse_loss(w, b)]
    for _ in range(n_steps):
        dw, db = mse_grads(w, b)
        w = w - lr * dw
        b = b - lr * db
        ws.append(w); bs.append(b); losses.append(mse_loss(w, b))
    return np.array(ws), np.array(bs), np.array(losses)


# Reproduce the lesson's first two hand-computed iterations.
w, b = 0.0, 0.0
print("start: w={:.3f} b={:.3f} L={:.4f}".format(w, b, mse_loss(w, b)))
for step in range(1, 3):
    dw, db = mse_grads(w, b)
    print("  iter {}: dL/dw={:.4f}  dL/db={:.4f}".format(step, dw, db))
    w, b = w - 0.1 * dw, b - 0.1 * db
    print("  iter {}: w={:.3f} b={:.3f} L={:.4f}".format(step, w, b, mse_loss(w, b)))

# Expected:
#   start : w=0.000 b=0.000 L=27.6667
#   iter 1: dL/dw=-22.6667  dL/db=-10.0000  ->  w=2.267 b=1.000 L=0.3319
#   iter 2: dL/dw=2.4889    dL/db=1.0667    ->  w=2.018 b=0.893 L=0.0053

In [ ]:
# Run the full fit and plot (1) the loss curve and (2) the path through (w, b) space.
ws, bs, losses = fit_line(lr=0.1, n_steps=50)
print("converged to w={:.4f}  b={:.4f}  L={:.2e}".format(ws[-1], bs[-1], losses[-1]))

# Loss surface over (w, b) for the contour background.
wg = np.linspace(-1, 4, 200)
bg = np.linspace(-2, 3, 200)
WG, BG = np.meshgrid(wg, bg)
LG = np.zeros_like(WG)
for i in range(WG.shape[0]):
    for j in range(WG.shape[1]):
        LG[i, j] = mse_loss(WG[i, j], BG[i, j])

fig, (ax_loss, ax_path) = plt.subplots(1, 2, figsize=(13, 5))

# (1) Loss curve
ax_loss.semilogy(losses, color='#818cf8', linewidth=2, marker='o', markersize=3)
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('MSE loss (log scale)')
ax_loss.set_title('Loss curve (eta = 0.1)', color='white')

# (2) Parameter path over the loss surface
cs = ax_path.contourf(WG, BG, LG, levels=30, cmap='YlOrRd', alpha=0.7)
plt.colorbar(cs, ax=ax_path, label='Loss')
ax_path.plot(ws, bs, '-o', color='#14b8a6', markersize=3, linewidth=1.6, label='GD path')
ax_path.scatter([2], [1], s=120, color='white', zorder=10, label='True (w, b)=(2, 1)')
ax_path.set_xlabel('w'); ax_path.set_ylabel('b')
ax_path.set_title('Parameter path', color='white')
ax_path.legend()

plt.tight_layout(); plt.show()

### The learning rate, concretely

Same line-fit problem, different step sizes. Edit the `lrs` list and re-run to
explore. Expect: `0.001` crawls, `0.1` converges smoothly, and `0.5` overshoots
and diverges to NaN (it explodes off the chart).

In [ ]:
# Compare learning rates on the line-fit MSE problem.
lrs = [0.001, 0.1, 0.5]
lr_colors = ['#64748b', '#14b8a6', '#f97316']

fig, ax = plt.subplots(figsize=(8, 5))
for lr, color in zip(lrs, lr_colors):
    _, _, losses = fit_line(lr=lr, n_steps=50)
    final = losses[-1]
    final_str = "{:.4g}".format(final) if np.isfinite(final) else "diverged"
    # Mask non-finite values so the divergent curve doesn't break the log plot.
    finite = np.where(np.isfinite(losses), losses, np.nan)
    ax.semilogy(finite, color=color, linewidth=2,
                label="lr={}  ->  L={}".format(lr, final_str))

ax.set_xlabel('Step'); ax.set_ylabel('MSE loss (log scale)')
ax.set_title('Effect of learning rate', color='white')
ax.legend()
plt.tight_layout(); plt.show()

## Optimizers on a harder surface

The line-fit bowl above is easy — gradient descent walks straight in. Real loss
surfaces have **ravines** (steep one way, flat the other) where plain SGD
struggles. To compare optimizers we switch to a 2D quadratic with asymmetric
curvature:

$$L(w_1, w_2) = 0.4 w_1^2 + 0.9 w_2^2 + 0.1 w_1 w_2$$

The global minimum is at $(0, 0)$. The different eigenvalues of the curvature
make it a good stress test for SGD, Momentum, RMSprop, and Adam.

In [ ]:
def loss(w):
    """2D quadratic bowl with asymmetric curvature."""
    w1, w2 = w
    return 0.4 * w1**2 + 0.9 * w2**2 + 0.1 * w1 * w2

def grad(w):
    """Analytical gradient of loss."""
    w1, w2 = w
    return np.array([0.8 * w1 + 0.1 * w2,
                     1.8 * w2 + 0.1 * w1])

# Plot the surface
w1s = np.linspace(-4, 4, 200)
w2s = np.linspace(-4, 4, 200)
W1, W2 = np.meshgrid(w1s, w2s)
Z = 0.4 * W1**2 + 0.9 * W2**2 + 0.1 * W1 * W2

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(W1, W2, Z, levels=25, cmap='YlOrRd', alpha=0.75)
plt.colorbar(cs, ax=ax, label='Loss')
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Loss Surface L(w₁, w₂)', color='white')
plt.tight_layout(); plt.show()

## Implementing optimizers from scratch

Each optimizer is a function `step(w, g, state) → (w_new, state_new)`.

In [ ]:
def sgd_step(w, g, state, lr=0.1):
    """Vanilla SGD: w ← w - lr * g"""
    return w - lr * g, state


def momentum_step(w, g, state, lr=0.1, beta=0.9):
    """SGD with Momentum: accumulates velocity."""
    v = state.get('v', np.zeros_like(w))
    v = beta * v + (1 - beta) * g
    return w - lr * v, {'v': v}


def rmsprop_step(w, g, state, lr=0.05, beta=0.9, eps=1e-8):
    """RMSprop: adaptive learning rates via squared gradient EMA."""
    s = state.get('s', np.zeros_like(w))
    s = beta * s + (1 - beta) * g**2
    return w - lr * g / (np.sqrt(s) + eps), {'s': s}


def adam_step(w, g, state, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam: combines momentum and RMSprop with bias correction."""
    t = state.get('t', 0) + 1
    m = state.get('m', np.zeros_like(w))
    v = state.get('v', np.zeros_like(w))
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)   # bias correction
    v_hat = v / (1 - beta2**t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps), {'t': t, 'm': m, 'v': v}


def run_optimizer(step_fn, w0, n_steps=60, **kwargs):
    w, state = np.array(w0, dtype=float), {}
    path = [w.copy()]
    losses = [loss(w)]
    for _ in range(n_steps):
        g = grad(w)
        w, state = step_fn(w, g, state, **kwargs)
        path.append(w.copy())
        losses.append(loss(w))
    return np.array(path), losses

In [ ]:
w0 = [3.5, 3.0]
results = {
    'SGD':      run_optimizer(sgd_step,      w0, lr=0.15),
    'Momentum': run_optimizer(momentum_step, w0, lr=0.1),
    'RMSprop':  run_optimizer(rmsprop_step,  w0, lr=0.12),
    'Adam':     run_optimizer(adam_step,     w0, lr=0.3),
}
colors = {'SGD': '#f97316', 'Momentum': '#818cf8', 'RMSprop': '#eab308', 'Adam': '#14b8a6'}

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(14, 5.5))

# Trajectory plot
ax_path.contourf(W1, W2, Z, levels=25, cmap='Greys', alpha=0.5)
for name, (path, _) in results.items():
    ax_path.plot(path[:, 0], path[:, 1], '-o', markersize=3,
                 color=colors[name], label=name, linewidth=1.8)
ax_path.scatter(0, 0, s=120, color='white', zorder=10, label='Minimum')
ax_path.set_title('Optimization Trajectories', color='white')
ax_path.legend()
ax_path.set_xlim(-4, 4); ax_path.set_ylim(-4, 4)

# Loss curves
for name, (_, losses) in results.items():
    ax_loss.semilogy(losses, color=colors[name], label=name, linewidth=2)
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('Loss (log scale)')
ax_loss.set_title('Convergence', color='white')
ax_loss.legend()

plt.tight_layout(); plt.show()

## Effect of learning rate

Too small = slow convergence. Too large = divergence.

In [ ]:
lrs = [0.01, 0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lr in zip(axes, lrs):
    path, losses = run_optimizer(sgd_step, w0, n_steps=80, lr=lr)
    ax.contourf(W1, W2, Z, levels=20, cmap='Greys', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], '-o', markersize=2.5, color='#818cf8', linewidth=1.5)
    ax.scatter(0, 0, s=80, color='#14b8a6', zorder=10)
    final = losses[-1]
    ax.set_title("lr={}  ->  L={:.4f}".format(lr, final), color='white', fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

plt.suptitle('SGD with Different Learning Rates', color='white', y=1.02)
plt.tight_layout(); plt.show()

## Momentum and the optimizer family

Plain SGD can crawl through ravines. **Momentum** accumulates a velocity that smooths the path; **Adam** adds per-parameter adaptive step sizes.

In [ ]:
def gd(grad, x0, lr=0.1, steps=50, momentum=0.0):
    x, v, path = x0, 0.0, [x0]
    for _ in range(steps):
        v = momentum * v - lr * grad(x)
        x = x + v
        path.append(x)
    return np.array(path)

# Minimize f(x) = x^2  (grad = 2x)
grad = lambda x: 2 * x
for m in [0.0, 0.9]:
    p = gd(grad, 5.0, lr=0.1, momentum=m)
    print("momentum={}: reached {:.4f} in {} steps".format(m, p[-1], len(p) - 1))

## Key takeaways

- Gradient descent steps **downhill**: $w \leftarrow w - \eta\,\nabla L$.
- The **learning rate** $\eta$ trades speed for stability — too big diverges, too small crawls.
- **Momentum** accelerates along consistent directions and damps oscillation.
- **Adam** (adaptive + momentum) is the safe default for deep learning.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — MSE gradients for a line fit

For the model $\hat{y} = wx + b$ with loss $L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$, the chain rule gives

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\,x_i, \qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

Implement both. The checks verify the gradients vanish at a perfect fit and match finite differences elsewhere — the exact pair of partial derivatives the worked example above stepped through.

In [ ]:
def mse_gradients(x, y, w, b):
    """Return (dL/dw, dL/db) for the MSE of the line w*x + b."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): the residuals (predictions minus targets)
    err = ...

    # TODO(you): dL/dw = 2 * mean(err * x)
    dw = ...

    # TODO(you): dL/db = 2 * mean(err)
    db = ...

    return dw, db

In [ ]:
# Checks — run me
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 5.0, 7.0])   # exactly y = 2x + 1

dw, db = mse_gradients(x, y, 2.0, 1.0)
assert abs(dw) < 1e-12 and abs(db) < 1e-12, "at the perfect fit, both gradients vanish"

def mse(x, y, w, b):
    return np.mean((w * x + b - y) ** 2)

w0, b0, h = 0.5, -0.3, 1e-6
dw, db = mse_gradients(x, y, w0, b0)
num_dw = (mse(x, y, w0 + h, b0) - mse(x, y, w0 - h, b0)) / (2 * h)
num_db = (mse(x, y, w0, b0 + h) - mse(x, y, w0, b0 - h)) / (2 * h)
assert abs(dw - num_dw) < 1e-5, "dL/dw must match the numerical gradient"
assert abs(db - num_db) < 1e-5, "dL/db must match the numerical gradient"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mse_gradients(x, y, w, b):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    err = w * x + b - y
    dw = 2 * np.mean(err * x)
    db = 2 * np.mean(err)
    return dw, db
```

</details>

### Exercise 2 — One momentum step

Momentum keeps a running **velocity** that accumulates gradients, then steps along it:

$$v \leftarrow \beta v + \nabla L(w), \qquad w \leftarrow w - \eta\, v$$

Implement a single update. The checks confirm the two facts that matter: $\beta = 0$ reduces to plain gradient descent, and a repeated gradient makes the velocity *grow* ($1 + \beta + \beta^2 + \dots$) — that's the "heavy ball" picking up speed down a consistent slope.

In [ ]:
def momentum_step(w, v, grad, lr=0.1, beta=0.9):
    """One momentum update. Returns the new (w, v)."""
    # TODO(you): update the velocity: beta * v + grad
    v = ...

    # TODO(you): step the weight against the velocity: w - lr * v
    w = ...

    return w, v

In [ ]:
# Checks — run me
w, v = momentum_step(1.0, 0.0, 0.5, lr=0.1, beta=0.0)
assert abs(w - 0.95) < 1e-12 and abs(v - 0.5) < 1e-12, "beta = 0 reduces to plain gradient descent"

_, v = momentum_step(*momentum_step(0.0, 0.0, 1.0, lr=0.1, beta=0.9), 1.0, lr=0.1, beta=0.9)
assert abs(v - 1.9) < 1e-12, "same gradient twice: v = 1 + 0.9 = 1.9 — velocity accumulates"

w, v = 10.0, 0.0
for _ in range(400):
    w, v = momentum_step(w, v, 2 * w, lr=0.05, beta=0.9)   # f(w) = w², grad = 2w
assert abs(w) < 1e-6, "momentum still converges on f(w) = w²"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def momentum_step(w, v, grad, lr=0.1, beta=0.9):
    v = beta * v + grad
    w = w - lr * v
    return w, v
```

</details>